# Foundations of ML Systems in Production

How to decide what an ML system should be before anyone builds it and the six-step methods

---

## **Agenda**

- [Preface](#scrollTo=c002)

1. [From a model to a system](#scrollTo=c003)
2. [Should this be ML at all](#scrollTo=c004)
3. [What breaks quietly](#scrollTo=c009)
4. [What every design is judged against](#scrollTo=c012)
5. [Framing the problem](#scrollTo=c013)
6. [Metrics that track the business](#scrollTo=c016)
7. [Competing objectives](#scrollTo=c019)
8. [How big is it](#scrollTo=c022)
9. [The design framework, and a live pass](#scrollTo=c025)
10. [Design synthesis and class summary](#scrollTo=c028)

- [References and further reading](#scrollTo=c029)

## **Preface**

The MLOps module ended on a claim. However a model is made, the second half of its life is the same. It gets versioned, approved, served and shared, and you can now do all of that.

This course starts one step earlier. Before anything is built, someone has to decide what to build.

- Should it be a model at all?
- What exactly should it predict?
- How will anyone know it works?
- How much traffic will it take?
- What happens when it is wrong?

Those decisions are **ML system design**. They are made at a whiteboard, mostly before any code, and they are expensive to change later.

One running example carries the whole class. A support team cannot keep up with its tickets, and asks for AI. The picture shows the path the class takes through that request.

![d0_class_map.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/621/original/d0_class_map.png?1794500485)

Read the top row from left to right, then the bottom row back. Each section answers one question about the ticket system. Section 9 turns those questions into six steps that work on any brief, and section 10 draws the whole design.


**By the end you will be able to do all of this.**

1. Decide whether a problem needs a lookup, a rule, a model, a bought service or a person.
2. Explain how production ML differs from research ML, and say where the hard decisions sit.
3. Recognise a silent failure and drift, and pick a fast signal and a slow signal that would catch them.
4. Write reliability, scalability, maintainability and adaptability requirements as targets you can test.
5. Frame a business request as an ML problem with inputs, an output, a label and an objective, and choose between classification, regression and ranking.
6. Choose offline, online and business metrics that track each other, and price errors by what they cost.
7. Decouple competing objectives so the business can rebalance them without retraining.
8. Make a back-of-envelope estimate of traffic, work and storage, and say what it rules out.
9. Drive an ambiguous brief through the six-step design framework.

**Prerequisites.** From the MLOps module, the ML lifecycle and its business framing stage, model decay, the "Hidden Technical Debt" paper, a prediction service, the model registry, and the model quality gate in CI/CD. From earlier study, classification, regression, precision, recall and logistic regression. Nothing about system design is assumed.

**Running the code.** The demonstrations use only numpy, pandas and scikit-learn. Colab already has all of them. On your own machine, install them into a fresh environment with `pip install numpy pandas scikit-learn`. The tickets are synthetic, built from a small set of sentence templates with a fixed seed. Real tickets are far more varied, so read each number as a picture of a design effect, not as a benchmark.

## **1. From a model to a system**

Start with a message that lands in a data team's inbox at a hypothetical online retailer. The head of customer support writes, "We get about 12,000 tickets a day. The queue is always behind, and customers wait two days for a first reply. Can we use AI on this?"

A data scientist could answer by the end of the afternoon. Export last quarter's tickets with the queue each one ended up in, and train a text classifier. Here a **queue** is the list of tickets waiting for one support team, such as returns or billing. Say the classifier picks the right queue for about 93% of tickets it never saw. That notebook is the easy part.

The hard part is everything the notebook never asked.

- How fast must a ticket be routed, and what may each routing cost?
- What happens when the model sends a hacked account to the returns queue?
- Who notices when a new product line brings tickets the model has never seen?
- Where do next month's training labels come from, and are they any good?
- Is wrong routing even the reason customers wait? If the team is simply short of agents, perfect routing saves little.

None of these questions is about the algorithm. All of them decide whether the system works.

### 1.1 Research ML and production ML

**Research ML** asks how good a model can get on a fixed data set. **Production ML** asks how to make a model useful to a business, every day, within real limits. The table follows Chip Huyen's comparison in her [Stanford CS 329S lecture notes](https://docs.google.com/document/d/1C3dlLmFdYHJmACVkz99lSTUPF4XQbWb_Ah7mPE12Igo). It shows where each side puts its weight.

| | Research | Production |
|---|---|---|
| The goal | The best score on a benchmark | Different goals for different people, which often pull apart |
| The data | Fixed and cleaned once | Constantly shifting, late and sometimes wrong |
| What must be fast | Training, so many experiments run | Answering one request, often in a fraction of a second |
| Fairness and explanations | Good to have | Important, because real people are affected |

The clearest story about the gap comes from Netflix. It once [offered a million dollars](https://web.archive.org/web/2013/http://techblog.netflix.com/2012/04/netflix-recommendations-beyond-5-stars.html) to anyone who could make its rating predictions 10% more accurate. A team won the first progress prize with an 8.43% improvement, built from 107 algorithms and more than 2,000 hours of work. Netflix put two of those algorithms into production. When the grand prize was won, Netflix tested the new methods offline and decided not to ship them. In its own words, the extra accuracy "did not seem to justify the engineering effort needed to bring them into a production environment."

**Source:** https://netflixtechblog.com/netflix-recommendations-beyond-the-5-stars-part-1-55838468f429#:~:text=did%20not%20seem%20to%20justify%20the%20engineering%20effort%20needed%20to%20bring%20them%20into%20a%20production%20environment.

### 1.2 Where the hard decisions sit

Here is the ticket system drawn as a system, with the model in its place.

![d1_system_around_model.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/622/original/d1_system_around_model.png?1794500488)

Read the top row from left to right. That is the **online path**, where a customer is waiting. A customer writes a ticket. The ticket service gathers what is known about it. The routing model returns a probability for each queue, and a decision step turns those numbers into an action. An agent resolves the ticket, or moves it to another queue.

Now read the bottom row from right to left. That is the **offline loop**, which runs in the background. Every resolved ticket lands in the ticket log. The queue that finally closed it becomes a label. A training job retrains the model, here once a week. The registry holds the versions that passed review, and only an approved version goes back into the online path.

You met the idea that ML code is a small box in a large system in the MLOps module. This course is about designing the rest of the boxes. The hard decisions fall in three places.

- **Data.** Which inputs exist at the moment of prediction, and where labels come from.
- **Serving.** How fast, how cheap, and what happens when the model is slow or down.
- **The feedback loop.** What the system learns from its own decisions. If an agent quietly handles a ticket the model sent to the wrong queue, the log records that queue as correct, and next week's model learns the mistake.

**Ask in the chat.** Swap the classifier for a large language model. Which of these three decisions go away? **Expected answer.** None of them. A bigger model still needs inputs it can see in time, a decision step, a fallback, and labels to check it against.

Before any of that, though, ask whether the model box belongs in the picture at all.

## **2. Should this be ML at all**

The support lead asked for AI. A good first answer is a question back. Which tickets could be handled without a model?

In plain words, **machine learning is worth building when a pattern is too complex to write down, there is data that shows it, and a wrong answer is cheap enough to accept.** Chip Huyen's (computer scientist and author of Designing Machine Learning Systems) notes add that the task should repeat, at scale, and that the pattern may keep changing. Her notes also give three reasons not to use ML. It is unethical, a simpler solution does the job, or it does not pay.

So there are five tools, and the cheaper ones come first.

- A **lookup** answers when the answer is a fact you already store. "Where is my order" needs the tracking status from the order database, not a prediction.
- A **rule** answers when the pattern is short and stable, such as "if the ticket says where is my order, send the tracking status."
- A **model** learns a pattern from past examples, such as which queue a ticket belongs in.
- You can **buy** a model when a problem is the same for many companies, such as turning phone calls into text. A vendor's model has seen far more data than yours. The price is a fee per call, data leaving the company, and a model that can change without notice.
- A **person** decides when a wrong answer is costly or the case is rare.

Google's [Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml) start in the same place. Rule 1 is "Don't be afraid to launch a product without machine learning." The picture puts the tools in the order to try them.

![d2_lookup_rule_model.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/623/original/d2_lookup_rule_model.png?1794500492)

Read the picture from left to right. Ask each question in turn and stop at the first yes. The last box matters too, because a person's decisions become the data a future model learns from.

Today "a model" can be a small classifier trained on your own tickets, or a large language model, usually bought, that reads the queue descriptions in a prompt. Choosing between them is model selection, later in the course. The system decisions in this class are the same for both. The demonstrations use the simplest option, weighted word counts fed to logistic regression, which runs in a second.

### 2.1 Testing the cheap tools on the tickets

Look at the tickets before deciding. The cell below builds a log of 20,000 tickets. They copy three things real ticket logs do. Some customers raise two issues in one ticket. Agents sometimes file a ticket under the wrong queue, here about 1 in 20. Some fraud tickets are vague, such as "something is wrong with my account", and read exactly like an ordinary account problem. Because the tickets are synthetic, each one also keeps its true queue, which a real log never has.

Then it tries six keyword rules, three that send the tracking status and three that send a password reset link, each wider than the one before.

**Before you run it.** Will adding keywords make a rule more useful, more risky, or both?

**[Please Note:]** Run this cell in the notebook. It takes about a second. The top half builds the tickets, and the cells in sections 3 and 6 build them the same way, so read from `log = tickets(...)` down.

In [ ]:
import numpy as np
import pandas as pd

# Synthetic support tickets, built from a fixed seed.
TEMPLATES = {
    "delivery": ["where is my order", "my parcel has not arrived", "tracking has not moved for days",
                 "order says delivered but nothing came", "when will my order arrive"],
    "returns":  ["how do I return the {i}", "I want to send back my {i}", "still waiting for my refund",
                 "refund not received for my return", "the {i} does not fit, can I return it"],
    "billing":  ["I was charged twice for my order", "my card payment failed", "the promo code did not apply",
                 "why was I charged a delivery fee", "the invoice amount is wrong",
                 "there is a charge I do not recognise"],
    "account":  ["I cannot log in", "the password reset email never came", "my account is locked",
                 "I forgot my password", "how do I change my email address",
                 "something is wrong with my account"],
    "damaged":  ["the {i} arrived broken", "my order arrived damaged", "the {i} is cracked",
                 "parts are missing from the {i}", "the box was crushed and the {i} is broken"],
    "fraud":    ["there are orders on my account I did not place", "someone changed my email address",
                 "I was charged for an order I never made", "my account was used without my permission",
                 "I got a password reset email I did not ask for",
                 "something is wrong with my account", "there is a charge I do not recognise"],
}
ITEMS = ["lamp", "kettle", "jacket", "chair", "blender", "rug"]
SHARE = [0.30, 0.22, 0.16, 0.14, 0.12, 0.06]


def tickets(n, seed):
    rng = np.random.default_rng(seed)
    queues = list(TEMPLATES)
    rows = []
    for k in rng.choice(len(queues), size=n, p=SHARE):
        options = TEMPLATES[queues[k]]
        text = options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        if rng.random() < 0.2:  # some customers raise a second issue in the same ticket
            options = TEMPLATES[queues[rng.integers(6)]]
            text += ". also " + options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        queue = queues[k] if rng.random() > 0.06 else queues[rng.integers(6)]  # agents sometimes file it wrong
        rows.append((text, queue, queues[k]))   # the true queue is kept for checking only
    return pd.DataFrame(rows, columns=["text", "queue", "true_queue"])


log = tickets(20_000, seed=0)
print(log.sample(6, random_state=4).to_string(index=False))
print()
print(log["queue"].value_counts(normalize=True).round(2).to_string())
print()

# Each automation is triggered by a keyword rule. Wider rules fire more often.
rules = [
    ("send tracking status", "delivery", "where is my order"),
    ("send tracking status", "delivery", "where is my order|tracking"),
    ("send tracking status", "delivery", "where is my order|tracking|arrive"),
    ("send password link", "account", "forgot my password"),
    ("send password link", "account", "forgot my password|log in|locked"),
    ("send password link", "account", "password|log in|locked|email"),
]

print(f"{'automation':22}{'keywords':36}{'fires on':>9}{'right':>8}{'fraud hits':>12}")
for action, queue, pattern in rules:
    fires = log["text"].str.contains(pattern)
    right = (log.loc[fires, "queue"] == queue).mean()
    fraud_hits = (log.loc[fires, "queue"] == "fraud").sum()   # tickets filed under fraud that the rule fires on
    print(f"{action:22}{pattern:36}{fires.mean():9.1%}{right:8.1%}{fraud_hits:12d}")


                              text   queue true_queue
something is wrong with my account   fraud      fraud
       still waiting for my refund returns    returns
            my card payment failed billing    billing
            the rug arrived broken damaged    damaged
       the invoice amount is wrong billing    billing
              my account is locked returns    account

queue
delivery    0.28
returns     0.22
billing     0.16
account     0.14
damaged     0.12
fraud       0.06

automation            keywords                             fires on   right  fraud hits
send tracking status  where is my order                        6.4%   88.2%          16
send tracking status  where is my order|tracking              12.8%   88.1%          35
send tracking status  where is my order|tracking|arrive       31.1%   72.6%         104
send password link    forgot my password                       2.8%   76.6%          14
send password link    forgot my password|log in|locked         8.6%   78

Read the six sample rows first. One fraud ticket says only "something is wrong with my account". The last row says "my account is locked". It was filed under returns, but its true queue is account, an agent's mistake. In the queue mix, delivery is the biggest at 28% of tickets and fraud the smallest at 6%.

Now the rules. "Right" means the log filed the ticket under that rule's queue, so a ticket with two issues counts as wrong even when the reply helped. The narrowest tracking rule fires on 6.4% of tickets and is right 88.2% of the time. Adding "tracking" doubles the reach to 12.8% and is still right 88.1% of the time. Adding "arrive" is where it breaks. The rule now fires on 31.1% of tickets but is right only 72.6% of the time, because "the rug arrived broken" is a damaged item, not a late one.

The fraud hits column is the one to worry about. Every rule fires on some tickets filed under fraud. For the tracking rules that costs little. The customer gets an extra tracking message, and the ticket stays open for an agent. For the password rules it is dangerous. A reset link goes to the account's email address, which an attacker may already have changed. Even the narrowest password rule fires on 14 fraud tickets, and the widest on 373. **A rule's mistakes are not spread evenly. The wider the rule, the more of its worst mistakes it makes.**

That points to a design. Keep the narrow tracking rules as automated replies. Often the reply answers the question before an agent reads the ticket, and a wrong one costs one extra message. Never let a keyword trigger an account action such as a password reset. Sending those tickets through a model does not make the action safe either. **Understanding a request and having permission to act on an account are separate decisions.** The account action needs its own identity checks, whatever routed the ticket. Google's [Rule 3](https://developers.google.com/machine-learning/guides/rules-of-ml) covers the wide rules. "Choose machine learning over a complex heuristic."

### Checkpoint 2.1: Lookup, rule, model, buy or a person

One more test helps with every choice in this section. Ask **who undoes a wrong answer, and how long it takes.** A wrong tracking message is undone in minutes, and a wrong account closure takes days of appeals.

The same retailer sends four more requests. 

**Q1.** Customers ask, "Has my refund gone through?" What should answer them?

- A. A model trained on past refund tickets
- B. A rule that replies "refunds take five to seven days"
- C. A bought chatbot that reads the ticket
- D. A lookup in the payments system

**Q2.** Tickets that mention legal action must reach a manager the same day. No past ticket is tagged as legal. What should flag them?

- A. A keyword rule, with a manager checking each flag
- B. A model trained before launch, since keywords miss some
- C. A lookup in the customer database
- D. A bought legal-risk model that also replies to the customer

**Q3.** About 600 tickets a day arrive in Spanish or Portuguese, an assumed number. Agents read only English. What should translate them?

- A. Your own translation model, trained on past tickets
- B. A rule that swaps words using a dictionary
- C. A bought translation service, tested on real tickets first
- D. Bilingual agents who translate each ticket by hand

**Q4.** The fraud team asks for accounts that look fraudulent to be closed automatically. What should you build?

- A. A model with 99% precision that closes them
- B. A model that ranks them, and a person who decides
- C. A keyword rule on words such as "hacked"
- D. A bought fraud model that closes them

**Why it matters.** The first design decision is which tool to use at all. A model chosen where a lookup would do is more expensive to run and less accurate than the database it replaces.

#### Discussion guide: Checkpoint 2.1

**Answers.** Q1 D, Q2 A, Q3 C, Q4 B. Reveal them one at a time, right after each question's replies come in.

1. **Q1, D. A lookup.** Refund status is a stored fact. A model (A) or a chatbot (C) can only guess what the database knows. B answers a different question, how long refunds take.
2. **Q2, A. A rule, checked by a person.** A short phrase list ("lawyer", "legal action", "court") catches many, and a manager reads every flag. B is the common wrong pick. No past ticket is tagged as legal, so there is nothing to train on. The manager's checked flags become those labels, and a model can be compared later. D answers a legal threat with no person in the loop.
3. **Q3, C. Buy, after a test.** Translation is the same problem for many companies, so a vendor's model has seen far more text than yours. Test it on real tickets, slang included. The costs are a fee per call and ticket text leaving the company. A needs far more text than a retailer has. B breaks on the first idiom. D works at ten tickets a day, not 600.
4. **Q4, B. A person, helped by a model.** A wrong answer locks a real customer out of their money, and the undo is slow, so the customer also needs a way to appeal. A sounds safe, but 99% precision means 1 closure in 100 hits an honest customer. C and D still close accounts with no person in the loop.

**Follow-up to post.** "The model in option A closes 2,000 accounts a week. How many honest customers is that each week?" The answer is 20, about a thousand a year, each locked out of their money.

## **3. What breaks quietly**

Say the team ships the router for the tickets the rules do not catch. The service dashboard stays green for weeks. Does that mean it works?

The MLOps module called this decay. Code is a bridge that stays standing, and a model is a map of a world that keeps changing. Here is what decay looks like from the inside. In plain words, **a silent failure is one where the system keeps answering, quickly and without errors, while its answers get worse.** Ordinary software tends to fail loudly, with a crash, an error code or a timeout. A wrong prediction looks exactly like a right one. Huyen's notes put it in four words, "ML systems fail silently."

The cell below trains the router once, on tickets from before a product launch. Then it replays a sample of 3,000 tickets from each of eight weeks. In the fourth week the retailer starts selling smart-home devices, and a new kind of ticket starts to arrive, such as "my smart plug will not connect." Agents open a new devices queue for them. The service itself is healthy the whole time, so its error count and latency are simulated as steady. The accuracy and the confidence come from the real model, with accuracy measured against the queue in the log.

The table has one column you have not met. **Low-confidence** is the share of tickets where the model's highest probability, across all queues, is below 0.5. It needs no labels, so it can be read the same day.

**Before you run it.** Which columns could move in the launch week, and which have to wait for labels?

**[Please Note:]** Run this cell in the notebook. It takes about two seconds. This is a deliberate failure. Before running it, collect a few answers to the question above from the chat and write them on the board.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# The same synthetic tickets, plus a new kind that starts arriving after a product launch in week 4.
TEMPLATES = {
    "delivery": ["where is my order", "my parcel has not arrived", "tracking has not moved for days",
                 "order says delivered but nothing came", "when will my order arrive"],
    "returns":  ["how do I return the {i}", "I want to send back my {i}", "still waiting for my refund",
                 "refund not received for my return", "the {i} does not fit, can I return it"],
    "billing":  ["I was charged twice for my order", "my card payment failed", "the promo code did not apply",
                 "why was I charged a delivery fee", "the invoice amount is wrong",
                 "there is a charge I do not recognise"],
    "account":  ["I cannot log in", "the password reset email never came", "my account is locked",
                 "I forgot my password", "how do I change my email address",
                 "something is wrong with my account"],
    "damaged":  ["the {i} arrived broken", "my order arrived damaged", "the {i} is cracked",
                 "parts are missing from the {i}", "the box was crushed and the {i} is broken"],
    "fraud":    ["there are orders on my account I did not place", "someone changed my email address",
                 "I was charged for an order I never made", "my account was used without my permission",
                 "I got a password reset email I did not ask for",
                 "something is wrong with my account", "there is a charge I do not recognise"],
    "devices":  ["my smart plug will not connect", "the app cannot find my camera", "how do I pair the doorbell",
                 "the speaker keeps disconnecting", "the thermostat will not connect to wifi"],
}
ITEMS = ["lamp", "kettle", "jacket", "chair", "blender", "rug"]
SHARE = [0.30, 0.22, 0.16, 0.14, 0.12, 0.06]


def tickets(n, seed, device_share=0.0):
    rng = np.random.default_rng(seed)
    queues = list(TEMPLATES)
    p = np.append(np.array(SHARE) * (1 - device_share), device_share)
    rows = []
    for k in rng.choice(len(queues), size=n, p=p):
        options = TEMPLATES[queues[k]]
        text = options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        if rng.random() < 0.2:
            options = TEMPLATES[queues[rng.integers(6)]]
            text += ". also " + options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        queue = queues[k] if rng.random() > 0.06 else queues[rng.integers(6)]
        rows.append((text, queue, queues[k]))
    return pd.DataFrame(rows, columns=["text", "queue", "true_queue"])


# Trained once, before launch, on tickets from before the new product existed.
train = tickets(6_000, seed=1)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
model = LogisticRegression(max_iter=2000).fit(vec.fit_transform(train["text"]), train["queue"])

rng = np.random.default_rng(7)
print("week  tickets  server errors  p95 latency  low-confidence  accuracy")
for week, new_share in enumerate([0, 0, 0, 0.04, 0.08, 0.12, 0.16, 0.20], start=1):
    batch = tickets(3_000, seed=100 + week, device_share=new_share)
    proba = model.predict_proba(vec.transform(batch["text"]))
    pred = model.classes_[proba.argmax(axis=1)]
    p95_ms = np.percentile(rng.normal(41, 4, len(batch)), 95)   # simulated, the service is healthy
    low_conf = (proba.max(axis=1) < 0.5).mean()
    known = week <= 6   # agents' final queues arrive about two weeks late
    acc = f"{(pred == batch['queue']).mean():8.1%}" if known else "  not known yet"
    print(f"{week:>4}{len(batch):>9,}{0:>15}{p95_ms:>10.0f} ms{low_conf:>16.1%}  {acc}")

# The last batch is the eighth week. How sure is the model about the new device tickets?
device = batch["true_queue"].to_numpy() == "devices"
print(f"\nWeek 8 device tickets scoring 0.7 or more on one of the old queues: {(proba.max(axis=1)[device] >= 0.7).mean():.0%}")

week  tickets  server errors  p95 latency  low-confidence  accuracy
   1    3,000              0        47 ms            1.4%     92.2%
   2    3,000              0        48 ms            1.3%     92.6%
   3    3,000              0        47 ms            1.5%     93.1%
   4    3,000              0        48 ms            2.8%     89.0%
   5    3,000              0        48 ms            4.6%     85.7%
   6    3,000              0        47 ms            5.2%     81.5%
   7    3,000              0        47 ms            7.6%    not known yet
   8    3,000              0        48 ms            8.2%    not known yet

Week 8 device tickets scoring 0.7 or more on one of the old queues: 42%


Read the table one column at a time, starting from the left. Server errors are zero every week. The p95 latency, the time within which 95 of every 100 requests finish, sits at 47 or 48 ms. A normal service dashboard would show eight green weeks in a row.

Now the accuracy column. It is 92.2%, 92.6% and 93.1% in the first three weeks, then 89.0%, 85.7% and 81.5% in the three weeks after the launch. The last two weeks are blank. Agents close tickets over days, so the final queue arrives about two weeks late. At the end of the eighth week, the newest accuracy anyone can see is from the sixth.

The column that moved on time is low-confidence. It reads 1.4%, 1.3% and 1.5% before the launch, then 2.8% in the launch week itself, and 8.2% by the eighth week. It nearly doubled in the launch week, without a single label. **When the label is late, watch what the model is unsure about.**

Now read the last line, because it limits that clue. In the eighth week, 42% of device tickets still score 0.7 or more on one of the six old queues. The model has no devices queue, so it forces each device ticket into a queue it knows, and often sounds sure about it. A confidence check at 0.7 would wave those tickets straight through to the wrong team. **Confidence is a clue, not a measure of quality.** A different change could leave it flat while accuracy falls.

### 3.1 Naming what went wrong

Four different things can change under a model, and each needs a different fix.

- **Data drift.** The inputs the model sees in production move away from the inputs it learned from. Here, new words arrived with a new product.
- **A new output.** The business added a queue the model cannot predict. That changes the task itself, so retraining on the old six queues cannot fix it.
- **Concept drift.** The inputs look the same, but the right answer changes. Suppose the retailer moves refunds for late deliveries from the returns queue to delivery. The tickets read exactly as before, so the model cannot see it. The business knows, but if nobody tells the model's owner, only the labels will.
- **A feedback loop,** from section 1, where the system's own decisions become its training labels. Device tickets that agents handle in the wrong queue, without moving them, would teach the next model to send them there again. In a feed, a loop like this can narrow what every user sees, which comes up in the next class.

**Ask in the chat.** Would the weekly retrain fix the devices problem on its own? **Expected answer.** No. Someone has to add the devices queue to the label set, collect enough labelled device tickets, and check the new model before it ships.

Measuring drift with statistical tests, and building full monitoring around it, comes later in the course. The design rule starts now, and the picture shows it.

![d7_two_signals.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/628/original/d7_two_signals.png?1794500507)

Read the top row first. A ticket is routed in seconds, but it can take up to two weeks to close, and only then does its label reach the log. A **fast signal** reads the model's output the same day and needs no labels, such as the low-confidence share or the mix of predicted queues. A **slow signal** waits for labels, such as accuracy, filled in as they arrive. A small sample of tickets checked by hand each week sits between the two. **Before launch, choose a fast signal and a slow signal for every model, and name who looks at them.**

## **4. What every design is judged against**

The router was up and wrong for weeks. So "the service is up" cannot be the whole requirement. An ML system needs requirements written in ML terms. Huyen groups them into four, [reliability, scalability, maintainability and adaptability](https://github.com/chiphuyen/dmls-book/blob/main/summary.md). Each has a plain meaning, and each fails in an ML-shaped way.

- **Reliability** means the system keeps doing the right thing at the agreed quality, even when parts of it fail. For an ML system, "the right thing" includes the quality of the answers, not only that an answer came back.
- **Scalability** means there is a sensible way to grow. The growth is not only in traffic. It is also in the number of models, such as one per language when the retailer opens in Spain, and in model size.
- **Maintainability** means people in different jobs can understand, change and rerun the system. The MLOps tools you know, tracking, the registry and CI/CD, exist to make this true.
- **Adaptability** means the system can follow changing data and changing business needs without a rebuild.

A requirement becomes testable once it has a target. The targets below are assumptions for this example.

| Requirement | A testable target for the router | How it fails quietly |
|---|---|---|
| Reliability | Catches at least 85% of fraud, and unsure tickets go to a person | Up and wrong, as in section 3 |
| Scalability | Still routes within 2 seconds at three times average traffic, and in a second language | A new language gets the English model's guesses |
| Maintainability | Another engineer can rerun training from versioned code and data | The person who knew the keyword list leaves |
| Adaptability | A new queue goes live within a week, through the normal review | Every change needs a full retrain and review |

### 4.1 Requirements need numbers

Three numbers come up in almost every design. **Latency** is how long one request takes. **Cost per prediction** is what one answer costs to compute. **Availability** is the share of requests the service answers at all.

Latency is stated as a **percentile**. In plain words, "p95 latency of 300 ms" means 95 of every 100 requests finish within 300 ms. Google's [SRE book](https://sre.google/sre-book/service-level-objectives/) explains why teams prefer percentiles to the average. "A simple average can obscure these tail latencies."

A worked example makes the point. Take 100 requests. Ninety finish in 40 ms and ten take 2,000 ms. The average is (90 × 40 + 10 × 2,000) / 100, which is 236 ms, and that sounds fine. The p95 is 2,000 ms, because the slowest ten requests include the 95th. One customer in ten waits two seconds. The average hides them, and the percentile shows them.

For the router, write the speed target as a percentile, 95% of tickets routed within 2 seconds. Add two more assumed targets. Spend under one cent per ticket on automated processing. Reach a routing decision for 99.9% of tickets each month, and save the rest for a person. That last target is about availability, and the fraud target is about quality. They are separate. A fallback needs capacity too. If one ticket in five goes to a person, that is 2,400 tickets a day someone must have time for.

**Ask in the chat.** Can the router meet its availability target and still fail its quality target? **Expected answer.** Yes. Section 3 did exactly that.

The requirements say how well the system must work. The next section decides what exactly it should predict.

## **5. Framing the problem**

"Use AI on our tickets" is a wish. In plain words, **framing turns a business request into an ML problem with defined inputs, an output, a label to learn from and an objective to optimise.** The MLOps module's business framing stage asked what decision a model improves and what metric proves it. Framing is the next step down, where that answer becomes something a model can be trained on.

### 5.1 From business objective to ML objective

Write the chain down, one link at a time.

| Link | For the router |
|---|---|
| The business objective | Cut the cost per ticket by 15% without lowering customer satisfaction. The 15% is an assumption. |
| The decision the system makes | Which queue receives each ticket, or whether a person triages it first |
| The ML objective | For each ticket, predict the probability that each queue is the one that resolves it |

Between the first two links sits a bet, that better routing lowers cost. Section 1 asked whether wrong routing is even why customers wait. Write the bet down and check it, for example by measuring how much agent time goes on moving tickets today. If agents spend most of their time writing replies, a model that suggests replies might save more.

Uber's customer support tool, [COTA](https://www.uber.com/blog/cota/), took that second path. Its models suggest the most likely issue types and solutions to an agent, who picks one. Uber ran an A/B test, which gives the new tool to a random group and compares it with a group without it. It reported that the tool reduced ticket handling time by about 10 percent. The model suggests and the agent decides, which keeps a person in the loop.

### 5.2 Inputs, outputs and labels

Everything here turns on one moment, when the prediction is made. For the router, that is when the ticket arrives. The picture splits the data at that moment.

![d8_moment_of_prediction.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/629/original/d8_moment_of_prediction.png?1794500510)

Read the left side first. **Inputs** are only what exists at the moment of prediction. For the router that is the ticket text, the channel, the customer's tier, their recent orders and the account's age. Now the right side. The agent's notes, the resolution code and the handling time are written later. A model trained with them looks brilliant offline and cannot run online, because those fields are empty when a new ticket arrives. This mistake is called **data leakage**, and the data class later in the course covers how to prevent it.

The **label** sits on the right too. It is the right answer the model learns from, here the queue that closed the ticket. That is a **natural label**, one the system's own records produce for free. Free labels have three catches.

- They are **noisy**, because agents misfile tickets.
- They are **late**, because closing a ticket takes days.
- They are **shaped by the system**, because they partly record where the router sent the ticket.

So pay for a few thousand tickets checked carefully by hand, spread across queues and languages, not only the easy ones. They make a clean test set. Section 6 shows how much that changes the picture.

The **output** is a probability for each queue, not just a queue name. A separate **decision step** turns it into an action. For example, route to the top queue when its probability is above 0.7, and otherwise send the ticket to a person. Keeping the output and the decision apart means the threshold can change without retraining. When there are several checks, such as a fraud check and a confidence check, their order is part of the decision too.

### 5.3 Choosing the formulation

The same request can be framed in more than one way, and the choice is a design decision. Three formulations cover most of this course.

- **Classification** outputs a category. Which queue should take this ticket?
- **Regression** outputs a number. How many minutes will this ticket take?
- **Ranking** outputs an order. Which ticket should an agent pick up next?

The picture lays the three side by side for the ticket request.

![d3_three_framings.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/624/original/d3_three_framings.png?1794500495)

Read each box from top to bottom. Each formulation asks a different question, needs a different label and is judged by a different metric. The log already has the closing queue and the handling time. Nobody records which tickets were urgent, so the ranking framing has a hidden cost, a new label someone must start collecting.

Two more choices are easy to miss. One ticket in five raises two issues, so a ticket could get two queues. That is **multi-label** classification, and it changes both the label and the metric. And a classifier with six fixed queues has no output for a seventh, so the devices queue needs a retrain. Scoring each pair of a ticket and a written queue description avoids that, at the price of one score per queue.

**[Please Note:]** Please name the moment of prediction before they list any inputs. It finds leakage faster than any talk about algorithms.

### Checkpoint 5.1: Frame a churn request

The head of support comes back with a new request. "Tell me which customers are about to leave us because of a bad support experience."

This request has a twist the router did not have. The router's label, the closing queue, arrives within about two weeks. Whether a customer left can only be known after a long wait to see if they come back. **When the label is slow, the training data is always old.**

Assume a retention team that can call 200 customers a day with an apology and a voucher. Post the four questions in the chat one at a time.

**Q1.** When should the prediction be made?

- A. When a support ticket closes
- B. When the customer cancels their account
- C. At the end of each quarter
- D. Whenever the customer opens the app

**Q2.** Which of these must **not** be an input?

- A. Contacts with support in the last 30 days
- B. Time to first reply on the ticket
- C. The customer's tier
- D. Whether they ordered again in the 60 days after the ticket

**Q3.** The team can make only 200 calls a day. Which formulation fits that decision best?

- A. Regression on the number of days until the next order
- B. A score for each customer whose ticket closed, top 200 called
- C. A classifier for the type of complaint in the ticket
- D. A leave-or-stay classifier, calling everyone above 0.5

**Q4.** The label is "no order in the 60 days after the ticket closed". To train on every customer from a ticket, not only the ones who came back, how old must that ticket be?

- A. About a day
- B. About a week
- C. At least 60 days
- D. It depends on the model

**Why it matters.** A request like this one sounds finished but is not. The framing decides whether anyone can act on the prediction, and how stale the training data will be.

#### Discussion guide: Checkpoint 5.1

**Answers.** Q1 A, Q2 D, Q3 B, Q4 C. Reveal each one before posting the next question.

1. **Q1, A. When a ticket closes.** That is the moment the bad experience has happened and someone can still act. B is too late, because the customer has already gone. C is too slow for an apology to mean anything. D has nothing to do with support.
2. **Q2, D.** Ordering again in the next 60 days is the label itself. It does not exist when the ticket closes. A model trained with it looks perfect offline and cannot run, which is the data leakage from section 5.2. A, B and C are all known at closing.
3. **Q3, B. A ranking.** Each day, score the customers whose tickets closed and sort them. The decision is "which 200 today", so the order is what matters. A classifier's probability, sorted from high to low, is a fine way to build it. D is the common wrong pick. A cut-off at 0.5 ignores the team's capacity. It could flag, for example, 3,000 customers on a bad day and 40 on a quiet one. A is possible but harder, because for customers who have not ordered yet the answer is still unknown. C predicts something nobody acts on.
4. **Q4, C. At least 60 days.** A customer who orders again has a "stayed" label within days. A customer only counts as gone once 60 days pass with no order. So recent tickets hold the early stayers and none of the leavers, and training on them makes leaving look rarer than it is. D sounds careful, but no model can learn from a label that has not arrived.

**Two limits to name after the reveal.** The label is a choice someone makes. A customer who orders twice a year will look like a leaver. And "likely to leave" differs from "would stay if called". Only a test that calls some customers and not others can measure that.

**Follow-up to post.** "Who does what differently on Monday because of this score?" Strong replies name the retention team, the 200 calls and the voucher.

## **6. Metrics that track the business**

The router is framed. How will anyone know it is good? Three kinds of metric answer that.

- An **offline metric** is computed on held-out data before launch. Accuracy, or recall for each queue. The model quality gate from the MLOps module checks one before release.
- An **online metric** is measured on live traffic after launch. The re-route rate, or the time to first reply.
- A **business metric** is what the company reports. Cost per ticket, or customer satisfaction.

The picture adds how long each kind usually takes to measure.

![d4_metric_ladder.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/625/original/d4_metric_ladder.png?1794500498)

Read the picture from left to right. You can compute the offline metric in minutes, so it is what you tune. The online metric takes days to weeks to settle. The business metric is reported each quarter. These speeds are typical, not fixed. Cost per ticket can be read daily, and an offline metric must wait for its labels too. Each arrow is a bet that the box on its left predicts the box on its right, and either bet can fail.

Both bets have failed in public. Netflix chose rating accuracy as a stand-in for what it wanted, and its own [account of the prize](https://web.archive.org/web/2013/http://techblog.netflix.com/2012/04/netflix-recommendations-beyond-5-stars.html) calls it a "proxy question". Its stated business objective was member satisfaction and subscription retention. YouTube found the same gap with clicks. In 2012 it [explained](https://blog.youtube/news-and-events/youtube-now-why-we-focus-on-watch-time/) that its discovery features had been "designed to drive views", which "rewarded videos that were successful at attracting clicks, rather than the videos that actually kept viewers engaged." It moved its target to watch time, which is still a stand-in, only a better one. Google's Rule 39 sums it up. Launch decisions are a proxy for long-term product goals.

### 6.1 Price the errors before you pick the metric

A metric that counts every error the same can pick the wrong design. The cell below trains one model and routes the same 6,000 test tickets two ways.

1. **Top queue.** Send each ticket to the queue with the highest probability.
2. **Fraud threshold.** The same model with a new decision step, from section 5. Send a ticket to fraud whenever its fraud probability is 0.2 or more, and otherwise to the top queue.

Retraining the model to weigh fraud tickets more heavily is another route, and the data class later in the course covers it. The cell scores each routing twice. **Log accuracy** compares with the queue in the log, agents' mistakes included. **True accuracy** compares with each ticket's true queue, which stands in for the hand-checked test set from section 5. Fraud recall and cost use the true queue. The mistakes are priced with two assumed costs. A wrong route costs \$2 of agent time to fix. A fraud ticket in the wrong queue costs \$150 instead, the average loss while an account takeover sits unhandled. The 0.2 is a choice for the demonstration. Picking a threshold from error costs, on separate validation tickets, comes back in the fraud case study.

**Before you run it.** Which routing will win on accuracy? Which will cost the least?

**[Please Note:]** Run this cell in the notebook. It takes about two seconds. Collect guesses from the chat and write them on the board first.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Synthetic support tickets, built from a fixed seed.
TEMPLATES = {
    "delivery": ["where is my order", "my parcel has not arrived", "tracking has not moved for days",
                 "order says delivered but nothing came", "when will my order arrive"],
    "returns":  ["how do I return the {i}", "I want to send back my {i}", "still waiting for my refund",
                 "refund not received for my return", "the {i} does not fit, can I return it"],
    "billing":  ["I was charged twice for my order", "my card payment failed", "the promo code did not apply",
                 "why was I charged a delivery fee", "the invoice amount is wrong",
                 "there is a charge I do not recognise"],
    "account":  ["I cannot log in", "the password reset email never came", "my account is locked",
                 "I forgot my password", "how do I change my email address",
                 "something is wrong with my account"],
    "damaged":  ["the {i} arrived broken", "my order arrived damaged", "the {i} is cracked",
                 "parts are missing from the {i}", "the box was crushed and the {i} is broken"],
    "fraud":    ["there are orders on my account I did not place", "someone changed my email address",
                 "I was charged for an order I never made", "my account was used without my permission",
                 "I got a password reset email I did not ask for",
                 "something is wrong with my account", "there is a charge I do not recognise"],
}
ITEMS = ["lamp", "kettle", "jacket", "chair", "blender", "rug"]
SHARE = [0.30, 0.22, 0.16, 0.14, 0.12, 0.06]


def tickets(n, seed):
    rng = np.random.default_rng(seed)
    queues = list(TEMPLATES)
    rows = []
    for k in rng.choice(len(queues), size=n, p=SHARE):
        options = TEMPLATES[queues[k]]
        text = options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        if rng.random() < 0.2:
            options = TEMPLATES[queues[rng.integers(6)]]
            text += ". also " + options[rng.integers(len(options))].format(i=ITEMS[rng.integers(6)])
        queue = queues[k] if rng.random() > 0.06 else queues[rng.integers(6)]
        rows.append((text, queue, queues[k]))
    return pd.DataFrame(rows, columns=["text", "queue", "true_queue"])


train, test = tickets(6_000, seed=1), tickets(6_000, seed=2)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train, X_test = vec.fit_transform(train["text"]), vec.transform(test["text"])

COST_REROUTE = 2         # dollars of agent time to move a ticket to the right queue (assumed)
COST_MISSED_FRAUD = 150  # dollars lost while a takeover sits in the wrong queue (assumed)

# The model learns from the noisy queue in the log, as a real team's would.
model = LogisticRegression(max_iter=2000).fit(X_train, train["queue"])
top_queue = model.predict(X_test)
p_fraud = model.predict_proba(X_test)[:, list(model.classes_).index("fraud")]
routes = {
    "top queue": top_queue,
    "fraud if P(fraud) >= 0.2": np.where(p_fraud >= 0.2, "fraud", top_queue),   # same model, new decision step
}
recorded = test["queue"].to_numpy()
truth = test["true_queue"].to_numpy()   # stands in for a hand-checked test set, used only to score
is_fraud = truth == "fraud"
print(f"{'routing':30}{'log acc.':>9}{'true acc.':>11}{'fraud recall':>14}{'to fraud':>10}{'cost':>10}")
for name, pred in routes.items():
    wrong = pred != truth
    missed_fraud = (wrong & is_fraud).sum()
    cost = (wrong.sum() - missed_fraud) * COST_REROUTE + missed_fraud * COST_MISSED_FRAUD
    print(f"{name:30}{(pred == recorded).mean():9.1%}{1 - wrong.mean():11.1%}"
          f"{1 - missed_fraud / is_fraud.sum():14.1%}{(pred == 'fraud').sum():10,d}{'$' + format(cost, ','):>10}")
print(f"\n{len(test):,} test tickets, {is_fraud.sum()} of them truly fraud")

routing                        log acc.  true acc.  fraud recall  to fraud      cost
top queue                         93.2%      98.1%         75.4%       273   $12,954
fraud if P(fraud) >= 0.2          89.1%      93.7%         98.0%       696    $1,792

6,000 test tickets, 349 of them truly fraud


Start with the two accuracy columns. Against the log, top-queue routing is right 93.2% of the time. Against the true queue it is right 98.1% of the time. Most of what the log calls the model's mistakes are the agents' mistakes. **A model can be better than its labels say,** and only a hand-checked set shows it. That is why section 5 said a clean test set is usually worth paying for.

Now the fraud column. Top-queue routing catches only 75.4% of the 349 true fraud tickets. The vague ones read exactly like account and billing tickets, and those queues are bigger, so the model sends them there. The threshold catches 98.0%. With the assumed costs, top-queue routing costs \$12,954 and the threshold costs \$1,792. **The more accurate routing costs about seven times as much.** It also misses the 85% fraud target from section 4. The fix needed no retraining, only a new decision step. That is the split between output and decision from section 5 paying off.

The to fraud column is the catch. The threshold sends 696 tickets to the fraud queue instead of 273, two and a half times as many, and about half of them are not fraud. The \$2 price assumes the fraud team can absorb them. If it cannot, real fraud waits longer in a fuller queue, which is the very cost the change was meant to cut. So check the fraud team's capacity before choosing.

### 6.2 Main metrics and guardrails

So choose the offline metric by what each error costs, such as the cost column above, or use accuracy with a floor on fraud recall. A floor like that is called a **guardrail**, a metric that must not get worse whatever happens to the main one. Give each level of the picture above a main metric and a guardrail.

- **Offline.** Cost-weighted error on hand-checked tickets, with fraud recall at or above 85% as the guardrail.
- **Online.** Hours until a fraud ticket reaches the fraud team, with the wait at the fraud queue and the triage desk as guardrails.
- **Business.** Cost per resolved ticket, with customer satisfaction as the guardrail.

Some numbers get worse on purpose. If agents move every wrongly routed ticket, the true accuracy column says the re-route rate rises from about 2% to about 6%, so agree its limit in advance. Comparing two systems on live traffic in this way is an A/B test. The next class designs one for a recommender, and its statistics come later in the course.

## **7. Competing objectives**

Routing decides which queue gets a ticket. Inside each queue there is a second decision, the order agents work in. The support lead wants two things from that order, and they pull against each other.

- Clear the backlog fast, which favours short tickets.
- Reach urgent customers fast, such as someone whose account was just taken over. Those tickets are often long.

**One option is a single model with a blended label.** Define priority as, say, 0.6 times urgency plus 0.4 times quickness, and train a model to predict it. It works until the business changes its mind. Every new balance means a new label, a new training run and a new evaluation. The balance itself is buried inside the model's weights, where nobody can see it.

**The other option is to decouple the objectives.** In plain words, **decoupling means predicting each objective separately, each from its own label, and combining the scores at serving time with a weight someone owns.** Huyen's [lecture notes](https://docs.google.com/document/d/10K3pYTNvreVy5hl2EqWf_LX3mMW4CQw1TdMrHplMu00) use a news feed with two models, one for post quality and one for engagement. Ranking by a weighted sum of their scores means "you can tweak α and β without retraining your models."

The picture shows the same split for the ticket queue.

![d5_decoupled.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/626/original/d5_decoupled.png?1794500500)

Read the picture from left to right. The ticket goes to two models. The urgency model returns the chance that the ticket is urgent. The handle-time model returns the expected minutes. The combine step turns minutes into a **quickness** score and mixes the two as w times urgency plus (1 - w) times quickness. The weight w comes in from above and belongs to the support lead.

Quickness needs a fixed scale. Here it is 1 / (1 + minutes / 15). A 5-minute ticket scores 0.75, a 15-minute ticket 0.5 and a 45-minute ticket 0.25. Because the 15 is fixed, the same ticket gets the same score however long the rest of the backlog is, so a weight of 0.4 means the same thing every morning.

Separate predictions do not always need separate models. One model with two outputs can share its inputs, with the weight still outside it, and choosing between the two is model selection. The urgency label is one that section 5 said nobody records yet. Agents can tag urgency at closing for a few weeks, and confirmed takeovers serve as a rougher label meanwhile.

The cell below takes a backlog of 400 tickets and four agents. The two models' outputs are simulated as noisy estimates of the truth, with no training. For each weight it orders the backlog, hands tickets to whichever agent is free first, and measures two things. How long urgent tickets wait on average, and how many tickets close in the first two hours.

**Before you run it.** What happens at w = 0, and at w = 1?

**[Please Note:]** Run this cell in the notebook. It finishes in under a second. Point out that only one line in the loop uses w, and nothing is retrained.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n, agents = 400, 4                      # a Monday-morning backlog and four agents (assumed)
urgent = rng.random(n) < 0.10           # the truth, unknown to the system
minutes = np.round(rng.lognormal(np.log(9), 0.6, n) * np.where(urgent, 1.6, 1.0))

# Two separately trained models, simulated. Each output is a noisy estimate of its own target.
p_urgent = np.clip(np.where(urgent, 0.7, 0.1) + rng.normal(0, 0.12, n), 0, 1)
est_minutes = minutes * rng.lognormal(0, 0.25, n)
quickness = 1 / (1 + est_minutes / 15)  # a fixed scale, so a score does not depend on the rest of the backlog

print(f"{n} tickets in the backlog, {urgent.sum()} of them urgent, {agents} agents\n")
print("weight w   average wait of urgent tickets   tickets closed in the first 2 hours")
for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
    order = np.argsort(-(w * p_urgent + (1 - w) * quickness))   # the only line that uses w
    free_at = np.zeros(agents)
    start = np.empty(n)
    for i in order:                     # the next ticket goes to whichever agent is free first
        a = free_at.argmin()
        start[i], free_at[a] = free_at[a], free_at[a] + minutes[i]
    closed = np.sum(start + minutes <= 120)
    print(f"{w:8.1f}   {start[urgent].mean():20.0f} minutes   {closed:24d}")

400 tickets in the backlog, 35 of them urgent, 4 agents

weight w   average wait of urgent tickets   tickets closed in the first 2 hours
     0.0                    653 minutes                         94
     0.2                    348 minutes                         91
     0.3                    188 minutes                         80
     0.4                     94 minutes                         56
     0.5                     65 minutes                         32
     0.7                     63 minutes                         25
     1.0                     70 minutes                         24


At w = 0 the queue runs shortest first. It closes 94 tickets in the first two hours, and the 35 urgent tickets wait 653 minutes on average, nearly 11 hours. The first step is nearly free. At 0.2 urgent tickets wait 348 minutes, 305 fewer, and only 3 fewer tickets close. At 0.3 they wait 188 minutes and 80 tickets close early. At 0.4 they wait 94 minutes and 56 close. At 0.5 they wait 65 minutes and 32 close.

Above 0.5 there is almost nothing left to gain. At 0.7 urgent tickets wait 63 minutes, only 2 fewer, and 7 fewer tickets close. At 1.0 urgent tickets wait longer, 70 minutes, with about the same closures, because the queue ignores length even among urgent tickets. **Every step from 0.2 to 0.5 is a business choice.** Going from 0.2 to 0.3 saves urgent customers 160 minutes and costs 11 early closures. From 0.3 to 0.4 saves 94 minutes for 24 closures. From 0.4 to 0.5 saves 29 minutes for another 24. Show the support lead this table and let them pick. Whatever they pick goes live by changing one number.

Decoupling has costs too. Two models mean two pipelines to run and monitor. And the weight only means something while each score keeps its meaning. If a retrained urgency model scores everything higher, a weight of 0.4 quietly behaves like 0.5. Keeping a score's meaning steady is called **calibration**, and it comes back when the course reaches offline evaluation. One blended model is the simpler choice when the business will never rebalance. And for the most serious tickets, a hard rule such as "no confirmed takeover waits more than 30 minutes" can beat any weight.

## Competing Objectives

Suppose the following tickets are waiting in a customer support queue.

| Ticket | Estimated Resolution Time | Urgency (0–1) |
|---|---:|---:|
| Account hacked | 40 min | 0.95 |
| Refund delayed | 15 min | 0.60 |
| Order status | 2 min | 0.10 |
| Update address | 3 min | 0.20 |
| Payment failure | 20 min | 0.80 |

The business has two competing objectives:

- **Maximise throughput:** Resolve as many tickets as possible by prioritising shorter tickets.
- **Minimise urgent customer waiting time:** Prioritise critical complaints, even if they take longer to resolve.

### Comparing the two strategies

| Metric | Quickest First | Most Urgent First |
|---|---:|---:|
| Tickets resolved in the first 20 minutes | 3 | 0 |
| Waiting time for hacked account | 40 min | 0 min |
| Short-term throughput | Higher | Lower |
| Responsiveness to urgent cases | Lower | Higher |

Improving one objective can negatively affect the other.

## Approach 1: Single Model with a Blended Label

Suppose management decides to assign 60% importance to urgency and 40% importance to quickness.

**Priority = 0.6 × Urgency + 0.4 × Quickness**

Quickness is a normalised score between 0 and 1, where a higher value indicates a shorter expected handling time.

### Training Dataset

| Ticket | Urgency | Quickness | Priority Label |
|---|---:|---:|---:|
| Account hacked | 0.95 | 0.20 | 0.65 |
| Refund delayed | 0.60 | 0.60 | 0.60 |
| Order status | 0.10 | 0.95 | 0.44 |
| Update address | 0.20 | 0.90 | 0.48 |
| Payment failure | 0.80 | 0.50 | 0.68 |

We train a regression model that directly predicts the priority score.

- **Input:** Ticket text and customer features.
- **Ground truth:** Blended priority label.
- **Output:** Predicted priority score.
- **Loss function:** Mean Squared Error (MSE).

The model does not independently predict urgency or quickness. It learns their combined value.

### What happens when the business objective changes?

Suppose fraud incidents increase, and management now wants to assign 90% importance to urgency and 10% to quickness.

**New Priority = 0.9 × Urgency + 0.1 × Quickness**

| Ticket | Old Priority Label | New Priority Label |
|---|---:|---:|
| Account hacked | 0.650 | 0.875 |
| Refund delayed | 0.600 | 0.600 |
| Order status | 0.440 | 0.185 |
| Update address | 0.480 | 0.270 |
| Payment failure | 0.680 | 0.770 |

The definition of the target variable has changed.

Consequently, we would generally need to:

1. Regenerate the training labels.
2. Retrain the model.
3. Evaluate the new model.
4. Deploy the new model.

The business policy is embedded inside the learned model parameters.

## Approach 2: Decoupling the Objectives

Instead of training a model to directly predict the combined priority score, train two independent predictors.

### Model 1: Urgency Prediction

| Ticket | Urgency Label |
|---|---:|
| Account hacked | 0.95 |
| Refund delayed | 0.60 |
| Order status | 0.10 |
| Update address | 0.20 |
| Payment failure | 0.80 |

### Model 2: Resolution Time Prediction

| Ticket | Resolution Time Label |
|---|---:|
| Account hacked | 40 min |
| Refund delayed | 15 min |
| Order status | 2 min |
| Update address | 3 min |
| Payment failure | 20 min |

The predicted resolution time can be converted into a normalised quickness score.

### Combining Predictions at Inference Time

**Priority = α × Predicted Urgency + β × Predicted Quickness**

Where:

- α represents the importance of urgency.
- β represents the importance of quickness.
- α + β = 1.

The weights are defined by the business policy rather than learned as part of the prediction models.

### Different Business Policies

| Business Policy | Urgency Weight | Quickness Weight |
|---|---:|---:|
| Maximise quick resolutions | 0.1 | 0.9 |
| Balanced strategy | 0.5 | 0.5 |
| Prioritise urgency | 0.9 | 0.1 |

### Comparing the Resulting Priority Scores

| Ticket | Quickness-Focused | Balanced | Urgency-Focused |
|---|---:|---:|---:|
| Account hacked | 0.275 | 0.575 | 0.875 |
| Refund delayed | 0.600 | 0.600 | 0.600 |
| Order status | 0.865 | 0.525 | 0.185 |
| Update address | 0.830 | 0.550 | 0.270 |
| Payment failure | 0.530 | 0.650 | 0.770 |

### Comparing the Resulting Ticket Order

| Rank | Quickness-Focused | Balanced | Urgency-Focused |
|---:|---|---|---|
| 1 | Order status | Payment failure | Account hacked |
| 2 | Update address | Refund delayed | Payment failure |
| 3 | Refund delayed | Account hacked | Refund delayed |
| 4 | Payment failure | Update address | Update address |
| 5 | Account hacked | Order status | Order status |

Notice that the underlying model predictions remain unchanged. Only the business policy changes.

No model retraining is required merely to adjust the relative importance of the two objectives, although the new policy still requires validation.

## Another Example: Social Media News Feed

A social media recommendation system has two competing objectives:

- **Engagement:** Recommend content that users are likely to click, like, share or comment on.
- **Content quality:** Recommend informative, trustworthy and valuable content.

| Content | Predicted Engagement | Predicted Quality |
|---|---:|---:|
| Celebrity gossip | 0.95 | 0.30 |
| Scientific article | 0.40 | 0.95 |
| Educational video | 0.70 | 0.85 |
| Clickbait article | 0.98 | 0.10 |

**Ranking Score = α × Engagement + β × Quality**

| Content | Engagement-Focused (90:10) | Balanced (50:50) | Quality-Focused (10:90) |
|---|---:|---:|---:|
| Celebrity gossip | 0.885 | 0.625 | 0.365 |
| Scientific article | 0.455 | 0.675 | 0.895 |
| Educational video | 0.715 | 0.775 | 0.835 |
| Clickbait article | 0.892 | 0.540 | 0.188 |

The company can change its recommendation policy without retraining its engagement and quality prediction models.

## Comparison of the Two Approaches

| Property | Single Blended Model | Decoupled Objectives |
|---|---|---|
| Changing business priorities | Generally requires retraining | Change decision weights |
| Explainability | Only combined score available | Individual scores available |
| Experimentation | Additional training runs | Adjust weights and evaluate |
| Debugging | Difficult to isolate objectives | Evaluate objectives independently |
| Adaptability | Lower | Higher |
| Inference complexity | Potentially lower | Potentially higher |
| Training labels | Combined priority label | Independent labels |

## Key Takeaway

- Machine learning models should predict the individual quantities relevant to a decision.
- The decision layer should determine the relative importance of those quantities.
- Separating predictions from business policies improves the adaptability of the ML system.
- Critical safety requirements may need hard constraints rather than weighted scores.

**The model predicts what is likely to happen. The business decides which outcomes are more desirable.**

Note: All datasets, scores and resolution times in this demonstration are synthetic and illustrative.

## **8. How big is it**

Everything so far decides what to predict and how to judge it. The other half of a design is how much work the system must do. In plain words, **a back-of-envelope estimate is a quick calculation with round numbers that tells you which designs are possible.** In the MLOps module, endpoint autoscaling added machines as traffic grew. The estimate tells you, before anything exists, whether you need one machine or a fleet. It only needs to be right to within a factor of two or three. Being wrong by ten times changes the design, so the estimate's job is to find the order of magnitude.

The steps are the same for any system.

1. Users times actions per user gives **requests per day**.
2. Divide by 86,400 seconds in a day, about 100,000, to get the **average requests per second**.
3. Multiply by a **peak factor**, often 2 to 3 for consumer apps, to get the rate in the busiest hour. This is an assumption, and real traffic can be far burstier.
4. Multiply by the work per request, such as how many items a model scores, to get **scores per second**. A score is a unit of work, not a separate call. A model server can score many items in one batch.
5. Multiply requests by the bytes logged for each one to get **storage per day**.

Work it once by hand for the router. The retailer ships 400,000 orders a day, and 3% of orders lead to a ticket. Both numbers are assumptions. That is 12,000 tickets a day. Divide by 86,400 and you get 0.14 tickets a second. At three times the average, the peak is 0.42 a second, or one ticket every 2.4 seconds in the busiest hour.

The cell below runs the same steps for the router and for a home feed in a hypothetical content app with 40 million daily users. Each user opens the feed 10 times a day, and each open scores 500 candidate items. The feed's numbers are the starting point of the next class.

**[Please Note:]** Run this cell in the notebook. It finishes instantly. Try to work out the feed's peak by hand, with the same peak factor of three. It is about 14,000 a second. After it runs, take one input to change from the chat, such as the peak factor, and rerun it.

In [ ]:
SECONDS_PER_DAY = 86_400


def estimate(name, requests_per_day, peak_factor, scores_per_request, kb_logged_per_request):
    average = requests_per_day / SECONDS_PER_DAY
    peak = average * peak_factor
    print(name)
    print(f"  requests per day           {requests_per_day:>15,.0f}")
    print(f"  requests per second, avg   {average:>15,.2f}")
    print(f"  requests per second, peak  {peak:>15,.2f}")
    print(f"  model scores per second    {peak * scores_per_request:>15,.2f}   at peak")
    print(f"  log written per day        {requests_per_day * kb_logged_per_request / 1e6:>15,.2f} GB")
    return requests_per_day, peak * scores_per_request


# Every input below is an assumption for a hypothetical retailer and app. Change them and rerun.
# Ticket triage. 400,000 orders a day, 3% become a ticket, one routing score each, 2 KB logged.
t_day, t_scores = estimate("Ticket triage", 400_000 * 0.03, 3, 1, 2)
print()
# A home feed. 40 million daily users open it 10 times, 500 candidates scored per open, 2 KB logged.
f_day, f_scores = estimate("Home feed", 40_000_000 * 10, 3, 500, 2)

print(f"\nA whole day of tickets is {t_day / (f_day / SECONDS_PER_DAY):.1f} seconds of feed traffic.")
print(f"At peak the feed computes {f_scores / t_scores:,.0f} times as many scores per second.")

# The 500 candidates assume something already narrowed the field. Without that step:
catalogue = 100_000_000   # items in the next class's app (assumed)
print(f"Scoring all {catalogue:,} items for every feed request: {f_scores / 500 * catalogue / 1e12:.1f} trillion scores a second.")

Ticket triage
  requests per day                    12,000
  requests per second, avg              0.14
  requests per second, peak             0.42
  model scores per second               0.42   at peak
  log written per day                   0.02 GB

Home feed
  requests per day               400,000,000
  requests per second, avg          4,629.63
  requests per second, peak        13,888.89
  model scores per second       6,944,444.44   at peak
  log written per day                 800.00 GB

A whole day of tickets is 2.6 seconds of feed traffic.
At peak the feed computes 16,666,667 times as many scores per second.
Scoring all 100,000,000 items for every feed request: 1.4 trillion scores a second.


The router handles 12,000 requests a day, 0.14 a second on average and 0.42 at peak, and writes 0.02 GB of log a day, about 24 MB. The feed handles 400 million requests a day, about 4,630 a second on average and 13,889 at peak. At 500 candidates each, it computes about 6.9 million model scores a second and writes 800 GB of log a day. **A whole day of tickets is 2.6 seconds of feed traffic.** That compares request counts only.

**For the router, compute is not the constraint.** At the ceiling of one cent per ticket from section 4, the whole day's automated budget is \$120. That is an allowance. The saving still has to be measured in agent time. Low traffic also does not prove the 2-second target. That depends on how long one ticket takes from end to end, so measure it, and if the model is a bought language model, measure it at busy times. The design effort belongs in labels, error costs and drift, which is where this class spent it.

**For the feed, compute is the first constraint.** Nearly 7 million scores a second, inside a deadline a user can feel, makes running a large model on every candidate very expensive. And those 500 candidates already assume that something narrowed the field. Scoring all 100 million items of the next class's catalogue would take 1.4 trillion scores a second. The estimate forces the design into stages, a cheap one that narrows the field and an expensive one that ranks what is left. That is where the next class starts.

Two habits help. Write each assumption next to its number, labelled. And sanity-check the result in another unit. For example, 800 GB a day is about 290 TB a year before compression or copies, which tells you how long the logs can be kept and what keeping them costs.

## **9. The design framework, and a live pass**

You have now done every step of a design once, in pieces, on one example. The **design framework** puts them in order, so the same steps work on any brief, at a whiteboard, in about 45 minutes.

![d6_design_framework.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/627/original/d6_design_framework.png?1794500503)

Read it in order around the board, from the top left. Each step names the sections where you already did it.

1. **Clarify.** Who uses it, what decision changes, what happens today, how fast, how often, where labels come from and what a wrong answer costs. You leave with requirements that have numbers, and with your assumptions written down. Sections 2, 4 and 5.
2. **Estimate.** Requests per second, data per day and cost. Section 8.
3. **Outline.** Boxes and arrows for the online path and the offline loop. Start with the simplest design that could work, including any lookup or rule in front of the model. Section 1.
4. **Deep dive.** The two hardest parts, where the numbers say the design is tight. For the router, that is the labels and the fraud queue, sections 5 and 6. For a feed, it is finding good candidates fast enough.
5. **Trade-offs.** For each major choice, the alternative and what you gave up. The fraud threshold against the fraud team's capacity, or one blended model against two. Sections 6 and 7.
6. **Failure modes.** What breaks quietly, how you would notice, with a fast signal and a slow one, and what the system does in the meantime. Section 3.

As a rule of thumb, steps 1 to 3 take the first third of the time and most of the rest goes to the deep dive. A common mistake is to spend half the time drawing boxes and run out before the hard part. When a later step breaks an assumption, go back and change it.

### 9.1 Driving an ambiguous brief

Real briefs arrive vague. The person asking usually knows the pain. Working out the problem is your job. **Driving a brief means you lead.** You ask the questions whose answers change the design, and when nobody knows an answer, you say an assumption out loud, write it down and move on. Three habits help.

- **Ask questions that change the design.** "How many?" changes the scale. "What is done today?" gives the baseline to beat. "When must the decision be made?" sets the speed. "What does a wrong answer cost?" picks the metric. "Do we ever learn if we were right?" tells you about labels. A question whose answer would not change the design is a question to skip.
- **Write assumptions where everyone can see them,** so anyone can challenge them.
- **Narrow early.** Pick one decision to improve first, and say out loud what you are leaving out.

Here is the brief for the live pass. A hypothetical grocery delivery company with 50 warehouses in 12 cities writes, "We throw away too much fresh food. Use ML to fix it."

**[Please Note:]** Plan the 27 minutes as about 5 on the framework above, 4 on clarifying, 14 on the checkpoint and the design on one page, and 4 on the follow-up. To clarify, ask the chat which questions the company must answer first. Write the useful ones on the board and answer each with a labelled assumption. The checkpoint below gives the assumptions and the decision to land on. Keep its worked answer collapsed until each question has had its replies.

### Checkpoint 9.1: A live pass on the fresh-food brief

Write these assumptions on the board, each labelled as an assumption. Each warehouse stocks about 3,000 fresh products. Buyers order every evening for the next morning, using the same day last week as their guide. Fresh products last two to five days, and about 8% of fresh stock is thrown away. Running out costs too, because a missing item loses the sale.

**The decision.** How many units of each fresh product each warehouse orders tonight. A buyer types the number in.

This brief has a trap the ticket router did not. The complaint is waste, but **the cheapest way to cut waste is to order less, and that empties the shelves.** A design that watches only one of the two numbers can look like a success while it hurts the business.

Now run steps 2, 3, 5 and 6 as a round of four questions. Step 4, the deep dive, is left for after class. Post each question in the chat, give a minute for letters, and reveal it before the next.

**Q1. Estimate.** There are 50 warehouses, each with 3,000 fresh products, and one order a night. How many forecasts does the system make each night?

- A. 3,000
- B. 150,000
- C. 1.8 million
- D. 55 million

**Q2. Outline and formulation.** What should the model predict, and what must it beat?

- A. Whether each product will be thrown away, beating random guesses
- B. A ranking of products by popularity, beating alphabetical order
- C. Tomorrow's units per product, beating a forecast of zero
- D. Tomorrow's units per product and warehouse, beating the buyers' order

**Q3. Trade-offs.** Waste and stockouts pull against each other. Where should that trade be set?

- A. In an ordering step the business owns, such as "cover demand on 90% of days"
- B. Nowhere, because a more accurate forecast makes both fall
- C. In the training loss, retrained whenever the business changes its mind
- D. By each buyer, by feel, each evening

**Q4. Failure modes.** Strawberries sell out at 2 pm every day. The sales log says 40 were sold. What goes wrong?

- A. Nothing, because customers wanted exactly 40
- B. The model orders too many strawberries, and waste goes up
- C. The model learns demand is 40 and sells out again, while waste looks excellent
- D. The nightly job runs out of time

**Why it matters.** This is the whole method on a brief nobody has framed for you. Every later class runs the same six steps on a harder problem.

#### Worked answer: Checkpoint 9.1

**Answers.** Q1 B, Q2 D, Q3 A, Q4 C. After each reveal, add that step to the design on the whiteboard, so the class builds one page together.

1. **Q1, B. 150,000 forecasts a night,** 50 times 3,000. C multiplies by the 12 cities too, but the cities already hold the 50 warehouses. D is a year of history, about 55 million rows. That is the training data, not the nightly work. **What the estimate rules out.** One shared model scores 150,000 rows in minutes on one machine, with no online latency at all. The requirement worth writing down, as an assumption, is that orders are ready an hour before the supplier's cutoff. So serving is easy, and the effort belongs to forecast quality and the ordering policy.
2. **Q2, D. Regression on tomorrow's units, against what buyers do today.** A model that cannot beat the buyers' order on the business metric does not ship. A tries to predict waste, but waste is a result of the order, not something to forecast. C is a baseline nobody uses, so beating it proves nothing. A forecast range beats a single number, because the order needs to know how unsure the forecast is. A later case study builds one.
3. **Q3, A. An ordering step the business owns.** This is the split between output and decision from section 5, with a weight owned by the business as in section 7. The setting can be higher for bread than for flowers, and it changes with no retraining. B is the most common wrong pick. A better forecast does shrink both waste and stockouts. But at any accuracy the forecast is sometimes over and sometimes under, and someone still has to pick which miss is cheaper. C works, but every change of mind becomes a retrain.
4. **Q4, C. The log hides demand.** You only ever see sales, never demand. When strawberries sell out at 2 pm, the log says "sold 40", not "wanted 70". A model trained on sales learns to order 40 again. This feedback loop is invisible in the waste metric. Catch it by logging when each product sold out, and treat a sold-out day as "at least 40", not "exactly 40". The fast signal is the share of products that sold out early, and the slow one is margin. If the data feed breaks, fall back to the old baseline. This is also why the target in Q2 should become demand rather than sales once sell-outs are logged.

**The design on one page.** At the evening cutoff, a batch job builds inputs from what is known then and forecasts tomorrow's demand. An ordering step turns each forecast into a quantity, using current stock, expiry dates and pack sizes. Buyers review the proposals, and their edits are logged. Sales, sell-outs and waste flow back as the next labels. Offline, the metric is forecast error weighted by each product's value, tested on weeks after the training data ends. Online, it is waste and stockouts per warehouse, in a pilot against warehouses that keep the old method. The business metric is the gross margin on fresh food.

**The deep dive, step 4,** is forecast quality and the ordering policy.

**Follow-up to post.** "What number does a buyer type in tonight, and how does this design change it?"

## **10. Design synthesis and class summary**

Here is the ticket system the class built, drawn in the order a ticket meets it.

**[Please Note:]** Zoom in on this picture before reading it, because its eleven boxes are small at notebook width.

![d9_final_design.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/252/630/original/d9_final_design.png?1794500514)

Follow one ticket through it.

1. **Lookups and narrow rules** answer what they can, then pass the ticket on. No rule acts on an account.
2. **The routing model** scores the ticket from what exists when it arrives.
3. **The fraud check comes first.** At 0.2 or more, a person on the fraud team checks the ticket.
4. **The confidence check comes next.** A top queue at 0.7 or more gets the ticket. Otherwise a person triages it, as they do whenever the model is down.
5. **Inside each queue,** urgency and quickness are combined with a weight the support lead owns.
6. **Every outcome is logged.** Closed tickets become labels for a weekly retrain, and only approved versions are deployed.
7. **Two signals watch it,** the low-confidence share daily, and accuracy as labels arrive.

The demonstrations tested each piece on its own. Before a pilot, test the whole chain on the same held-out tickets, and check that the fraud team and the triage desk can take the load. Section 3 showed that the 0.7 check alone will not stop a new kind of ticket.

| Choice | What it buys | What it costs |
|---|---|---|
| Rules in front of the model | Cheap, explainable answers for easy tickets | Someone must own them, and wide rules fail in the worst place |
| A fraud threshold of 0.2 | 98% of fraud caught, not 75%, with no retraining | Two and a half times the fraud queue's inflow |
| Two scores and a weight | Priorities change without retraining | Two pipelines, and scores that must keep their meaning |
| Labels from the ticket log | Free and plentiful | Noisy, about two weeks late, shaped by the router |

**A model is one box in a system.** Its hard decisions sit in the data, the serving path and the feedback loop, and it is judged on latency, cost and reliability as well as accuracy.

**Try a lookup and a rule before a model.** Narrow rules are cheap and mostly right. Widening them is where they break, and that is where a model earns its cost. Neither a rule nor a model gets to act on an account alone.

**ML systems fail quietly.** Before launch, choose a fast signal that needs no labels and a slow one that does. Confidence is a clue, not proof.

**Requirements need ML meanings and numbers.** Reliability includes the quality of answers, and latency is a percentile, never an average.

**Framing comes before modelling.** Name the decision and the moment of prediction, use only inputs that exist then, find a label you can collect, and pick the formulation that fits.

**Pick metrics by what errors cost.** The more accurate routing cost about seven times as much, and a new threshold fixed it without retraining. A hand-checked test set showed the model was better than its log said.

**Decouple objectives the business will rebalance.** Then a change of priorities means changing one number.

**Estimate before you design, and drive the brief.** A day of tickets is 2.6 seconds of feed traffic. Six steps, with your assumptions on the board, turn a vague wish into a design.

## **References and further reading**

**The method**

- [Chip Huyen, Designing Machine Learning Systems, chapter summaries](https://github.com/chiphuyen/dmls-book/blob/main/summary.md). The four requirements and much of this class's structure.
- [Chip Huyen, CS 329S lecture notes, Machine Learning Systems in Production](https://docs.google.com/document/d/1C3dlLmFdYHJmACVkz99lSTUPF4XQbWb_Ah7mPE12Igo). When to use ML, and research against production.
- [Chip Huyen, CS 329S lecture notes, ML and Data Systems Fundamentals](https://docs.google.com/document/d/10K3pYTNvreVy5hl2EqWf_LX3mMW4CQw1TdMrHplMu00). Decoupling objectives, with a news feed.
- [Martin Zinkevich, Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml). Start with rules 1 to 4 and 38 to 39.

**Production stories**

- [Netflix, Netflix Recommendations, Beyond the 5 stars, Part 1 (2012, archived copy)](https://web.archive.org/web/2013/http://techblog.netflix.com/2012/04/netflix-recommendations-beyond-5-stars.html). Why the prize-winning model was not deployed.
- [YouTube, Why We Focus on Watch Time (2012)](https://blog.youtube/news-and-events/youtube-now-why-we-focus-on-watch-time/). Moving the target from clicks to time watched.
- [Uber, COTA, Improving Uber Customer Care with NLP and Machine Learning (2018)](https://www.uber.com/blog/cota/). A ticket assistant like this class's router, with its A/B test.
- [Molino, Zheng and Wang, COTA, the paper (2018)](https://arxiv.org/abs/1807.01337). The models and the experiment in detail.

**Reliability and the system around the model**

- [Google SRE book, Service Level Objectives](https://sre.google/sre-book/service-level-objectives/). Percentiles, targets and why averages mislead.
- [Sculley et al., Hidden Technical Debt in Machine Learning Systems (NeurIPS 2015)](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html). Worth rereading with a designer's eye.